#Loading Libraries

In [1]:
YOUR_NGROK_AUTH_TOKEN = ""

In [2]:
import nltk
nltk.download('wordnet', quiet=True)
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
import http.server
import socketserver
import threading
from pyngrok import ngrok

In [3]:
!pip install flask flask-ngrok joblib

In [4]:
!pip install PyPDF2

In [5]:
!pip install textstat

In [ ]:
from flask import Flask, request, jsonify, send_file
import os
from pyngrok import ngrok
from nltk.tokenize import sent_tokenize, word_tokenize
import PyPDF2
import io
from collections import Counter
import textstat
import pandas as pd
import joblib
import pickle
from IPython.core.display import Javascript

In [ ]:
!pip install sumy
!pip install transformers

from sumy.parsers.plaintext import PlaintextParser
from sumy.nlp.tokenizers import Tokenizer
from sumy.summarizers.lsa import LsaSummarizer
from sumy.nlp.stemmers import Stemmer
from sumy.utils import get_stop_words
import torch
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

In [ ]:
import re
from spacy.lang.en import English
from spacy.lang.en.stop_words import STOP_WORDS
from string import punctuation
import spacy
from nltk.stem import WordNetLemmatizer

nlp = spacy.load("en_core_web_sm", disable=["parser", "ner"])
lemmatizer = WordNetLemmatizer()

#20Newsgroup

In [6]:
# Load the dataset with topic names
try:
    df_topics = pd.read_csv('/content/Data Analysed/processed_newsgroups_dataset_with_topic_names.csv')
    print("Loaded df_topics.csv successfully. First 5 rows:")
    display(df_topics.head())
    print("Columns:", df_topics.columns.tolist())
except FileNotFoundError:
    print("Error: 'processed_newsgroups_dataset_with_topic_names (1).csv' not found. Please ensure the file exists.")
    df_topics = None

# Load the dataset with sentiment information
try:
    df_sentiment_dataset = pd.read_csv('/content/Data Analysed/processed_newsgroups_dataset_after_sentiment.csv')
    print("\nLoaded df_sentiment_dataset.csv successfully. First 5 rows:")
    display(df_sentiment_dataset.head())
    print("Columns:", df_sentiment_dataset.columns.tolist())
except FileNotFoundError:
    print("Error: 'processed_newsgroups_dataset_after_sentiment.csv' not found. Please ensure the file exists.")
    df_sentiment_dataset = None

Loaded df_topics.csv successfully. First 5 rows:


,text,summary,category_name,lda_categorical_topic,lda_topic_name,lda_simplified_topic_name
0,sure bashers pen fan pretty confused lack kind...,I am sure some bashers of Pens fans are pretty...,"rec, sport, hockey",23,"game, team, entry, year, player",Gaming
1,brother market high performance video card sup...,My brother is in the market for a high-perform...,"comp, sys, ibm, pc, hardware",29,"window, card, problem, driver, color",Windows
2,finally said dream mediterranean new area grea...,Finally you said what you dream about. Mediter...,"talk, politics, mideast",27,"armenian, year, turkish, muslim, people",Armenian Conflict
3,think scsi card dma transfer disk scsi card dm...,Think! It's the SCSI card doing the DMA transf...,"comp, sys, ibm, pc, hardware",14,"drive, disk, scsi, hard, controller",Disk Drive
4,old jasmine drive use new system understanding...,1) I have an old Jasmine drive which I cannot ...,"comp, sys, mac, hardware",14,"drive, disk, scsi, hard, controller",Disk Drive


Columns: ['text', 'summary', 'category_name', 'lda_categorical_topic', 'lda_topic_name', 'lda_simplified_topic_name']

Loaded df_sentiment_dataset.csv successfully. First 5 rows:


,text,summary,category_name,lda_categorical_topic,lda_topic_name,sentiment_scores,compound_sentiment,sentiment_category
0,sure bashers pen fan pretty confused lack kind...,I am sure some bashers of Pens fans are pretty...,"rec, sport, hockey",23,"game, team, entry, year, player","{'neg': 0.28, 'neu': 0.424, 'pos': 0.296, 'com...",0.1531,Positive
1,brother market high performance video card sup...,My brother is in the market for a high-perform...,"comp, sys, ibm, pc, hardware",29,"window, card, problem, driver, color","{'neg': 0.035, 'neu': 0.759, 'pos': 0.206, 'co...",0.7430,Positive
2,finally said dream mediterranean new area grea...,Finally you said what you dream about. Mediter...,"talk, politics, mideast",27,"armenian, year, turkish, muslim, people","{'neg': 0.362, 'neu': 0.51, 'pos': 0.128, 'com...",-0.9876,Negative
3,think scsi card dma transfer disk scsi card dm...,Think! It's the SCSI card doing the DMA transf...,"comp, sys, ibm, pc, hardware",14,"drive, disk, scsi, hard, controller","{'neg': 0.0, 'neu': 0.808, 'pos': 0.192, 'comp...",0.9100,Positive
4,old jasmine drive use new system understanding...,1) I have an old Jasmine drive which I cannot ...,"comp, sys, mac, hardware",14,"drive, disk, scsi, hard, controller","{'neg': 0.0, 'neu': 0.911, 'pos': 0.089, 'comp...",0.5574,Positive


Columns: ['text', 'summary', 'category_name', 'lda_categorical_topic', 'lda_topic_name', 'sentiment_scores', 'compound_sentiment', 'sentiment_category']


#UI


In [7]:
html_content = """
<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<title>AI Text Analysis Dashboard</title>
<link rel="stylesheet" href="styles.css">
<script src="https://cdn.jsdelivr.net/npm/chart.js"></script>
<script>
// Chart.js Plugin for center text - Registered once globally
const centerTextPlugin = {
    id: 'centerText',
    beforeDraw: function(chart) {
        if (chart.config.options.elements && chart.config.options.elements.center) {
            const ctx = chart.ctx;
            const canvas = chart.canvas;

            const centerConfig = chart.config.options.elements.center;
            const fontStyle = centerConfig.fontStyle || 'Arial';
            const txt = centerConfig.text;
            const color = centerConfig.color || '#000';
            const sidePadding = centerConfig.sidePadding || 20;
            const sidePaddingCalculated = (sidePadding / 100) * (canvas.width / 2);

            ctx.font = 'bold 20px ' + fontStyle;
            ctx.fillStyle = color;
            ctx.textAlign = 'center';
            ctx.textBaseline = 'middle';

            const centerX = (canvas.width / 2);
            const centerY = (canvas.height / 2);

            ctx.fillText(txt, centerX, centerY);
        }
    }
};

// Register the plugin globally once
Chart.register(centerTextPlugin);

async function analyze() {
    const loader = document.getElementById("loader");
    loader.classList.remove("hidden");

    const fileInput = document.getElementById("fileInput").files[0];
    const textInput = document.getElementById("textInput").value;
    const errorMessageDiv = document.getElementById("errorMessage");
    errorMessageDiv.textContent = ''; // Clear previous errors

    let formData = new FormData();

    if (fileInput) {
        const fileName = fileInput.name;
        const fileExtension = fileName.split('.').pop().toLowerCase();
        if (fileExtension !== 'txt' && fileExtension !== 'pdf') {
            errorMessageDiv.textContent = 'Only .txt and .pdf files are allowed.';
            loader.classList.add("hidden");
            return; // Stop analysis if file type is invalid
        }
        formData.append("file", fileInput);
    } else if (textInput.trim() !== '') {
        formData.append("text", textInput);
    } else {
        errorMessageDiv.textContent = 'Please upload a file or paste some text.';
        loader.classList.add("hidden");
        return; // Stop analysis if no input
    }

    const response = await fetch("/analyze", {
        method: "POST",
        body: formData
    });

    const data = await response.json();

    console.log('API Response Data:', data); // DEBUG

    loader.classList.add("hidden");

    if (data.error) {
        errorMessageDiv.textContent = data.error;
    } else {
        const dominantSentiment = displayResults(data);
        drawSentimentDoughnutChart(data.sentiments, dominantSentiment); // Call for doughnut chart
        drawSentimentBarChart(data.sentiments); // Call for bar chart
        drawWordFrequencyChart(data.top_words);
        drawBigramFrequencyChart(data.top_bigrams);
        drawSentenceLengthChart(data.sentence_length_distribution);
        drawWordLengthChart(data.word_length_distribution);
        displayReadabilityScores(data.readability);

        document.getElementById('tabs').classList.remove('hidden'); // Show tabs after analysis
        openTab('insightsContent'); // Automatically open insights tab
    }
}

function displayResults(data) {
    document.getElementById("insightsResults").innerHTML = `
        <p><b>Topics:</b> ${data.topics.join(", ")}</p>
        <p><b>Abstractive Summary:</b> ${data.abstractive_summary}</p>
        <p><b>Extractive Summary:</b> ${data.extractive_summary}</p>
    `;

    // Calculate and display dominant sentiment
    const sentimentCounts = { positive: 0, negative: 0, neutral: 0 };
    data.sentiments.forEach(s => sentimentCounts[s]++);

    let dominantSentiment = 'Neutral'; // Default if no sentiments
    const sentimentsArray = Object.entries(sentimentCounts);

    if (sentimentsArray.length > 0) {
        // Sort by count descending to find the most frequent sentiment
        sentimentsArray.sort((a, b) => b[1] - a[1]);
        dominantSentiment = sentimentsArray[0][0];
        // Capitalize the first letter for display
        dominantSentiment = dominantSentiment.charAt(0).toUpperCase() + dominantSentiment.slice(1);
    }

    const sentimentDisplayDiv = document.createElement('p');
    sentimentDisplayDiv.innerHTML = `<b>Overall Sentiment: ${dominantSentiment}</b>`;
    document.getElementById("insightsResults").prepend(sentimentDisplayDiv); // Add at the top of results
    return dominantSentiment; // Return dominant sentiment
}

/* 📊 Animated Doughnut Chart */
function drawSentimentDoughnutChart(sentiments, dominantSentiment) {
    console.log('drawSentimentDoughnutChart called with sentiments:', sentiments, 'dominantSentiment:', dominantSentiment); // DEBUG
    const counts = { positive: 0, negative: 0, neutral: 0 };
    sentiments.forEach(s => counts[s]++);
    console.log('Sentiment counts for doughnut chart:', counts); // DEBUG

    let chartStatus = Chart.getChart("sentimentDoughnutChart");
    if (chartStatus != undefined) {
        chartStatus.destroy();
    }

    new Chart(document.getElementById("sentimentDoughnutChart"), {
        type: "doughnut",
        data: {
            labels: ["Positive", "Negative", "Neutral"],
            datasets: [{
                data: [counts.positive, counts.negative, counts.neutral],
                backgroundColor: ['#4CAF50', '#F44336', '#FFEB3B'], // Custom colors
                hoverOffset: 4
            }]
        },
        options: {
            responsive: true,
            maintainAspectRatio: false, // Add this line
            plugins: {
                legend: {
                    position: 'top',
                    labels: {
                        color: 'white' // Make legend labels visible on dark background
                    }
                },
                title: {
                    display: true,
                    text: 'Sentiment Distribution',
                    color: 'white' // Make title visible
                }
            },
            // Removed elements.center configuration
            animation: {
                animateScale: true,
                animateRotate: true
            }
        }
    });
}

/* 📊 Animated Bar Chart for Sentiments */
function drawSentimentBarChart(sentiments) {
    console.log('drawSentimentBarChart called with sentiments:', sentiments); // DEBUG
    const counts = { positive: 0, negative: 0, neutral: 0 };
    sentiments.forEach(s => counts[s]++);
    console.log('Sentiment counts for bar chart:', counts); // DEBUG

    let chartStatus = Chart.getChart("sentimentBarChart");
    if (chartStatus != undefined) {
        chartStatus.destroy();
    }

    new Chart(document.getElementById("sentimentBarChart"), {
        type: 'bar',
        data: {
            labels: ["Positive", "Negative", "Neutral"],
            datasets: [{
                label: 'Number of Sentences',
                data: [counts.positive, counts.negative, counts.neutral],
                backgroundColor: ['#4CAF50', '#F44336', '#FFEB3B'],
                borderColor: ['#4CAF50', '#F44336', '#FFEB3B'],
                borderWidth: 1
            }]
        },
        options: {
            responsive: true,
            maintainAspectRatio: false, // Add this line
            plugins: {
                legend: {
                    display: false // No need for legend in simple bar chart
                },
                title: {
                    display: true,
                    text: 'Sentiment Counts Breakdown',
                    color: 'white' // Make title visible
                }
            },
            scales: {
                y: {
                    beginAtZero: true,
                    ticks: {
                        color: 'white', // Y-axis labels color
                        precision: 0 // Ensure integer ticks for counts
                    },
                    grid: {
                        color: 'rgba(255, 255, 255, 0.2)' // Y-axis grid lines color
                    }
                },
                x: {
                    ticks: {
                        color: 'white' // X-axis labels color
                    },
                    grid: {
                        color: 'rgba(255, 255, 255, 0.2)' // X-axis grid lines color
                    }
                }
            }
        }
    });
}

/* 📊 Bar Chart for Word Frequency */
function drawWordFrequencyChart(topWords) {
    console.log('drawWordFrequencyChart called with:', topWords); // DEBUG
    const labels = topWords.map(item => item[0]);
    const data = topWords.map(item => item[1]);

    let chartStatus = Chart.getChart("wordFrequencyChart");
    if (chartStatus != undefined) {
        chartStatus.destroy();
    }

    new Chart(document.getElementById("wordFrequencyChart"), {
        type: 'bar',
        data: {
            labels: labels,
            datasets: [{
                label: 'Word Frequency',
                data: data,
                backgroundColor: 'rgba(75, 192, 192, 0.6)',
                borderColor: 'rgba(75, 192, 192, 1)',
                borderWidth: 1
            }]
        },
        options: {
            responsive: true,
            maintainAspectRatio: false,
            plugins: {
                legend: { display: false },
                title: { display: true, text: 'Top 10 Most Frequent Words', color: 'white' }
            },
            scales: {
                y: { beginAtZero: true, ticks: { color: 'white', precision: 0 }, grid: { color: 'rgba(255, 255, 255, 0.2)' } },
                x: { ticks: { color: 'white' }, grid: { color: 'rgba(255, 255, 255, 0.2)' } }
            }
        }
    });
}

/* 📊 Bar Chart for Bigram Frequency */
function drawBigramFrequencyChart(topBigrams) {
    console.log('drawBigramFrequencyChart called with:', topBigrams); // DEBUG
    const labels = topBigrams.map(item => item.bigram);
    const data = topBigrams.map(item => item.count);

    let chartStatus = Chart.getChart("bigramFrequencyChart");
    if (chartStatus != undefined) {
        chartStatus.destroy();
    }

    new Chart(document.getElementById("bigramFrequencyChart"), {
        type: 'bar',
        data: {
            labels: labels,
            datasets: [{
                label: 'Bigram Frequency',
                data: data,
                backgroundColor: 'rgba(153, 102, 255, 0.6)',
                borderColor: 'rgba(153, 102, 255, 1)',
                borderWidth: 1
            }]
        },
        options: {
            responsive: true,
            maintainAspectRatio: false,
            plugins: {
                legend: { display: false },
                title: { display: true, text: 'Top 10 Most Frequent Bigrams', color: 'white' }
            },
            scales: {
                y: { beginAtZero: true, ticks: { color: 'white', precision: 0 }, grid: { color: 'rgba(255, 255, 255, 0.2)' } },
                x: { ticks: { color: 'white' }, grid: { color: 'rgba(255, 255, 255, 0.2)' } }
            }
        }
    });
}

/* 📊 Bar Chart for Sentence Length Distribution */
function drawSentenceLengthChart(sentenceLengthData) {
    console.log('drawSentenceLengthChart called with:', sentenceLengthData); // DEBUG
    const labels = sentenceLengthData.map(item => item.label);
    const data = sentenceLengthData.map(item => item.count);

    let chartStatus = Chart.getChart("sentenceLengthChart");
    if (chartStatus != undefined) {
        chartStatus.destroy();
    }

    new Chart(document.getElementById("sentenceLengthChart"), {
        type: 'bar',
        data: {
            labels: labels,
            datasets: [{
                label: 'Number of Sentences',
                data: data,
                backgroundColor: 'rgba(255, 159, 64, 0.6)',
                borderColor: 'rgba(255, 159, 64, 1)',
                borderWidth: 1
            }]
        },
        options: {
            responsive: true,
            maintainAspectRatio: false,
            plugins: {
                legend: { display: false },
                title: { display: true, text: 'Sentence Length Distribution', color: 'white' }
            },
            scales: {
                y: { beginAtZero: true, ticks: { color: 'white', precision: 0 }, grid: { color: 'rgba(255, 255, 255, 0.2)' } },
                x: { ticks: { color: 'white' }, grid: { color: 'rgba(255, 255, 255, 0.2)' } }
            }
        }
    });
}

/* 📊 Bar Chart for Word Length Distribution */
function drawWordLengthChart(wordLengthData) {
    console.log('drawWordLengthChart called with:', wordLengthData); // DEBUG
    const labels = wordLengthData.map(item => item.label);
    const data = wordLengthData.map(item => item.count);

    let chartStatus = Chart.getChart("wordLengthChart");
    if (chartStatus != undefined) {
        chartStatus.destroy();
    }

    new Chart(document.getElementById("wordLengthChart"), {
        type: 'bar',
        data: {
            labels: labels,
            datasets: [{
                label: 'Number of Words',
                data: data,
                backgroundColor: 'rgba(255, 99, 132, 0.6)',
                borderColor: 'rgba(255, 99, 132, 1)',
                borderWidth: 1
            }]
        },
        options: {
            responsive: true,
            maintainAspectRatio: false,
            plugins: {
                legend: { display: false },
                title: { display: true, text: 'Word Length Distribution', color: 'white' }
            },
            scales: {
                y: { beginAtZero: true, ticks: { color: 'white', precision: 0 }, grid: { color: 'rgba(255, 255, 255, 0.2)' } },
                x: { ticks: { color: 'white' }, grid: { color: 'rgba(255, 255, 255, 0.2)' } }
            }
        }
    });
}

/* Display Readability Scores */
function displayReadabilityScores(readabilityData) {
    console.log('displayReadabilityScores called with:', readabilityData); // DEBUG
    const readabilityDiv = document.getElementById('readabilityScores');
    readabilityDiv.innerHTML = `
        <p><b>Flesch Reading Ease:</b> ${readabilityData.flesch_reading_ease.toFixed(2)}</p>
        <p><b>Flesch-Kincaid Grade Level:</b> ${readabilityData.flesch_kincaid_grade.toFixed(2)}</p>
    `;
}

function updateFileName(input) {
    const fileNameSpan = document.getElementById('fileName');
    if (input.files && input.files.length > 0) {
        fileNameSpan.textContent = input.files[0].name;
    }
 else {
        fileNameSpan.textContent = "No file chosen";
    }
}

function removeFile() {
    const fileInput = document.getElementById('fileInput');
    fileInput.value = ''; // Clear the file input
    document.getElementById('fileName').textContent = 'No file chosen'; // Reset file name display
}

// Tab switching functionality
function openTab(tabName, clickedEvent) {
    var i, tabContent, tabButtons;

    tabContent = document.getElementsByClassName("tab-content");
    for (i = 0; i < tabContent.length; i++) {
        tabContent[i].style.display = "none";
    }

    tabButtons = document.getElementsByClassName("tab-button");
    for (i = 0; i < tabButtons.length; i++) {
        tabButtons[i].classList.remove("active");
    }

    document.getElementById(tabName).style.display = "block";

    if (clickedEvent && clickedEvent.currentTarget) {
        clickedEvent.currentTarget.classList.add("active");
    } else {
        // If called programmatically, find the corresponding button and activate it
        const targetButton = document.querySelector(`.tab-buttons button[onclick*="openTab('${tabName}')"]`);
        if (targetButton) {
            targetButton.classList.add('active');
        }
    }

    // Special handling for the 20Newgroup tab to load and render charts
    if (tabName === 'newgroupContent') {
        loadAndRender20NewsgroupCharts();
    }
}

// New functions for 20 Newsgroup Visualizations
async function loadAndRender20NewsgroupCharts() {
    try {
        const response = await fetch('/get_20newsgroup_data');
        const data = await response.json();

        if (data.error) {
            console.error('Error loading 20 Newsgroup data:', data.error);
            document.getElementById('newgroupContent').innerHTML = `<p style="color: red;">${data.error}</p>`;
            return;
        }

        console.log('20 Newsgroup API Data:', data);

        draw20NewsgroupTopicDistribution(data.topic_distribution);
        draw20NewsgroupOverallSentiment(data.overall_sentiment_distribution);
        // draw20NewsgroupSentimentByTopic(data.sentiment_by_topic); // Commented out as per user's request

    } catch (error) {
        console.error('Failed to fetch 20 Newsgroup data:', error);
        document.getElementById('newgroupContent').innerHTML = `<p style="color: red;">Failed to load visualizations. Please check server status.</p>`;
    }
}

function draw20NewsgroupTopicDistribution(topicData) {
    console.log('Drawing Topic Distribution:', topicData);
    const labels = Object.keys(topicData);
    const data = Object.values(topicData);

    let chartStatus = Chart.getChart("topicDistributionChart");
    if (chartStatus != undefined) {
        chartStatus.destroy();
    }

    new Chart(document.getElementById("topicDistributionChart"), {
        type: 'bar',
        data: {
            labels: labels,
            datasets: [{
                label: 'Number of Documents',
                data: data,
                backgroundColor: 'rgba(54, 162, 235, 0.6)',
                borderColor: 'rgba(54, 162, 235, 1)',
                borderWidth: 1
            }]
        },
        options: {
            responsive: true,
            maintainAspectRatio: false,
            plugins: {
                legend: { display: false },
                title: { display: true, text: '20 Newsgroup Topic Distribution', color: 'white' }
            },
            scales: {
                y: { beginAtZero: true, ticks: { color: 'white', precision: 0 }, grid: { color: 'rgba(255, 255, 255, 0.2)' } },
                x: { ticks: { color: 'white', autoSkip: false, maxRotation: 90, minRotation: 45 }, grid: { color: 'rgba(255, 255, 255, 0.2)' } }
            }
        }
    });
}

function draw20NewsgroupOverallSentiment(sentimentData) {
    console.log('Drawing Overall Sentiment:', sentimentData);
    const labels = Object.keys(sentimentData);
    const data = Object.values(sentimentData);
    const backgroundColors = labels.map(label => {
        if (label === 'Positive') return '#4CAF50';
        if (label === 'Negative') return '#F44336';
        if (label === 'Neutral') return '#FFEB3B';
        return '#CCCCCC';
    });

    let chartStatus = Chart.getChart("overallSentimentChart");
    if (chartStatus != undefined) {
        chartStatus.destroy();
    }

    new Chart(document.getElementById("overallSentimentChart"), {
        type: 'doughnut',
        data: {
            labels: labels,
            datasets: [{
                data: data,
                backgroundColor: backgroundColors,
                hoverOffset: 4
            }]
        },
        options: {
            responsive: true,
            maintainAspectRatio: false,
            plugins: {
                legend: { position: 'top', labels: { color: 'white' } },
                title: { display: true, text: '20 Newsgroup Overall Sentiment Distribution', color: 'white' }
            }
        }
    });
}

function draw20NewsgroupSentimentByTopic(sentimentByTopicData) {
    console.log('Drawing Sentiment by Topic:', sentimentByTopicData);
    const topics = Object.keys(sentimentByTopicData);
    const sentiments = ['Positive', 'Negative', 'Neutral'];

    const datasets = sentiments.map(sentiment => {
        return {
            label: sentiment,
            data: topics.map(topic => sentimentByTopicData[topic][sentiment] || 0),
            backgroundColor: sentiment === 'Positive' ? '#4CAF50' :
                             sentiment === 'Negative' ? '#F44336' :
                             '#FFEB3B',
            borderColor: sentiment === 'Positive' ? '#4CAF50' :
                         sentiment === 'Negative' ? '#F44336' :
                         '#FFEB3B',
            borderWidth: 1
        };
    });

    let chartStatus = Chart.getChart("sentimentByTopicChart");
    if (chartStatus != undefined) {
        chartStatus.destroy();
    }

    new Chart(document.getElementById("sentimentByTopicChart"), {
        type: 'bar',
        data: {
            labels: topics,
            datasets: datasets
        },
        options: {
            responsive: true,
            maintainAspectRatio: false,
            plugins: {
                legend: { position: 'top', labels: { color: 'white' } },
                title: { display: true, text: '20 Newsgroup Sentiment Distribution by Topic', color: 'white' }
            },
            scales: {
                x: {
                    stacked: true,
                    ticks: { color: 'white', autoSkip: false, maxRotation: 90, minRotation: 45 },
                    grid: { color: 'rgba(255, 255, 255, 0.2)' }
                },
                y: {
                    stacked: true,
                    beginAtZero: true,
                    ticks: { color: 'white', precision: 0 },
                    grid: { color: 'rgba(255, 255, 255, 0.2)' }
                }
            }
        }
    });
}

</script>
</head>

<body>

<div class="background"></div>

<div class="dashboard">
    <h1>AI Text Intelligence</h1>

    <!-- Input Panel -->
    <div class="card glass">
        <h2>Upload or Paste Text</h2>
        <div class="file-input-wrapper">
            <label for="fileInput" class="custom-file-upload">
                <input type="file" id="fileInput" onchange="updateFileName(this)" accept=".txt,.pdf">
                <span id="fileName">No file chosen</span>
                <span class="upload-button">Choose File</span>
            </label>
            <button class="remove-file-button" onclick="removeFile()">Clear File</button>
        </div>
        <textarea id="textInput" placeholder="Paste your text..."></textarea>
        <button onclick="analyze()">Analyze 🚀</button>
        <div id="errorMessage" style="color: red; margin-top: 10px;"></div>
    </div>

    <!-- Loader -->
    <div id="loader" class="loader hidden"></div>

    <!-- Tabs -->
    <div id="tabs" class="tab-container hidden">
        <div class="tab-buttons">
            <button class="tab-button active" onclick="openTab('insightsContent', event)">Insights</button>
            <button class="tab-button" onclick="openTab('graphsContent', event)">Graphs</button>
            <button class="tab-button" onclick="openTab('newgroupContent', event)">20Newgroup Visualizations</button>
        </div>

        <!-- Insights Tab Content -->
        <div id="insightsContent" class="tab-content card glass">
            <h2>Insights</h2>
            <div id="insightsResults"></div>
        </div>

        <!-- Graphs Tab Content -->
        <div id="graphsContent" class="tab-content card glass" style="display:none;">
            <h2>Sentiment Analytics</h2>
            <div class="charts-container">
                <div style="width: 350px; height: 350px;">
                    <canvas id="sentimentDoughnutChart"></canvas>
                </div>
                <div style="width: 350px; height: 350px;">
                    <canvas id="sentimentBarChart"></canvas>
                </div>
            </div>

            <h2>Text Composition Analytics</h2>
            <div class="charts-container">
                <div style="width: 450px; height: 350px;">
                    <canvas id="wordFrequencyChart"></canvas>
                </div>
                <div style="width: 450px; height: 350px;">
                    <canvas id="bigramFrequencyChart"></canvas>
                </div>
            </div>

            <div class="charts-container">
                <div style="width: 450px; height: 350px;">
                    <canvas id="sentenceLengthChart"></canvas>
                </div>
                <div style="width: 450px; height: 350px;">
                    <canvas id="wordLengthChart"></canvas>
                </div>
            </div>

            <h2>Readability Scores</h2>
            <div id="readabilityScores" class="readability-container">
                <!-- Readability scores will be displayed here -->
            </div>

        </div>

        <!-- 20Newgroup Visualizations Tab Content -->
        <div id="newgroupContent" class="tab-content card glass" style="display:none;">
            <h2>20 Newsgroup Visualizations</h2>
            <img src="/sentiment_per_topic_image" alt="Sentiment per Topic" style="max-width: 100%; height: auto; margin-top: 20px; margin-bottom: 20px; border-radius: 8px; box-shadow: 0 4px 8px rgba(0,0,0,0.2);">
            <div class="charts-container">
                <div style="width: 450px; height: 350px;">
                    <canvas id="topicDistributionChart"></canvas>
                </div>
                <div style="width: 350px; height: 350px;">
                    <canvas id="overallSentimentChart"></canvas>
                </div>
            </div>
            <!-- Removed sentimentByTopicChart as per user's request -->
            <!-- <div class="charts-container">
                <div style="width: 90%; height: 400px;">
                    <canvas id="sentimentByTopicChart"></canvas>
                </div>
            </div> -->
        </div>
    </div>


</body>
</html>
"""
css_content = """
body {
    margin: 0;
    font-family: 'Segoe UI', sans-serif;
    color: white;
    overflow-x: hidden;
}

/* 🌈 Animated background */
.background {
    position: fixed;
    width: 100%;
    height: 100%;
    background: linear-gradient(-45deg, #0ea5e9, #9333ea, #06b6d4, #3b82f6);
    background-size: 400% 400%;
    animation: gradient 12s ease infinite;
    z-index: -1;
}

@keyframes gradient {
    0% {background-position: 0%}
    50% {background-position: 100%}
    100% {background-position: 0%}
}

.dashboard {
    width: 80%;
    margin: auto;
    padding: 30px;
}

h1 {
    text-align: center;
}

/* 🧢 Glassmorphism */
.glass {
    background: rgba(255, 255, 255, 0.1);
    border-radius: 16px;
    padding: 20px;
    margin: 20px 0;
    backdrop-filter: blur(10px);
    box-shadow: 0 8px 32px rgba(0,0,0,0.3);
}

/* Inputs */
textarea {
    width: 100%;
    height: 100px;
    margin-top: 10px;
    border-radius: 10px;
    padding: 10px;
}

/* 🚀 Button animation */
button {
    margin-top: 10px;
    padding: 10px 20px;
    border-radius: 10px;
    border: none;
    background: linear-gradient(45deg, #06b6d4, #3b82f6);
    color: white;
    cursor: pointer;
    transition: 0.3s;
}

button:hover {
    transform: scale(1.05);
    box-shadow: 0 0 15px #06b6d4;
}

/* ⚡ Loader */
.loader {
    border: 6px solid rgba(255,255,255,0.2);
    border-top: 6px solid #06b6d4;
    border-radius: 50%;
    width: 50px;
    height: 50px;
    animation: spin 1s linear infinite;
    margin: 20px auto;
}

.hidden {
    display: none;
}

@keyframes spin {
    100% { transform: rotate(360deg); }
}

/* File input styling */
.file-input-wrapper {
    display: flex;
    align-items: center;
    gap: 10px; /* Space between file upload and remove button */
    margin-top: 10px;
}

.custom-file-upload {
    display: flex;
    align-items: center;
    background: rgba(255, 255, 255, 0.1);
    border-radius: 10px;
    padding: 8px;
    flex-grow: 1; /* Allow it to take available space */
    cursor: pointer;
    border: 1px solid rgba(255, 255, 255, 0.2);
    transition: background 0.3s ease;
}

.custom-file-upload:hover {
    background: rgba(255, 255, 255, 0.2);
}

.custom-file-upload input[type="file"] {
    display: none; /* Hide the default file input */
}

#fileName {
    flex-grow: 1;
    color: rgba(255, 255, 255, 0.7);
    margin-left: 10px;
    white-space: nowrap;
    overflow: hidden;
    text-overflow: ellipsis;
}

.upload-button {
    background: linear-gradient(45deg, #06b6d4, #3b82f6);
    color: white;
    padding: 8px 15px;
    border-radius: 8px;
    margin-left: 10px;
    white-space: nowrap;
    transition: transform 0.3s ease, box-shadow 0.3s ease;
}

.custom-file-upload:hover .upload-button {
    transform: scale(1.05);
    box-shadow: 0 0 10px rgba(6, 182, 212, 0.7);
}

.remove-file-button {
    margin-top: 0; /* Override default button margin-top */
    background: linear-gradient(45deg, #f44336, #e57373); /* Red gradient */
    padding: 8px 15px; /* Match custom-file-upload button padding */
    border-radius: 8px;
    transition: 0.3s;
}

.remove-file-button:hover {
    transform: scale(1.05);
    box-shadow: 0 0 10px rgba(244, 67, 54, 0.7);
}

/* Tab Styles */
.tab-container {
    margin-top: 20px;
}

.tab-buttons {
    display: flex;
    gap: 10px;
    margin-bottom: 10px;
}

.tab-button {
    background: rgba(255, 255, 255, 0.1);
    border: 1px solid rgba(255, 255, 255, 0.2);
    color: white;
    padding: 10px 20px;
    border-radius: 8px;
    cursor: pointer;
    transition: background 0.3s ease, transform 0.3s ease;
    margin-top: 0; /* Override default button margin */
}

.tab-button:hover {
    background: rgba(255, 255, 255, 0.2);
    transform: translateY(-2px);
}

.tab-button.active {
    background: linear-gradient(45deg, #06b6d4, #3b82f6);
    border-color: #06b6d4;
    transform: translateY(-2px);
    box-shadow: 0 4px 15px rgba(6, 182, 212, 0.4);
}

.tab-content {
    /* Styles for the content inside each tab */
    padding: 20px;
    border-radius: 16px;
    background: rgba(255, 255, 255, 0.1);
    backdrop-filter: blur(10px);
    box-shadow: 0 8px 32px rgba(0,0,0,0.3);
}

.charts-container {
    display: flex;
    flex-wrap: wrap; /* Allow items to wrap to the next line if space is limited */
    justify-content: center; /* Center charts horizontally */
    gap: 20px; /* Space between charts */
    margin-top: 20px;
}

"""

with open("index.html", "w", encoding='utf-8') as f:
    f.write(html_content)
with open("styles.css", "w", encoding='utf-8') as f:
    f.write(css_content)
print("index.html created successfully.")

index.html created successfully.


#Cleaning models loading

In [8]:
lda_model = joblib.load('/content/Models/lda_model.joblib')
sentiment_model = pickle.load(open('/content/Models/sentiment_analyzer.pkl','rb'))

dct = {
    0: 'Software',
    1: 'Bikes',
    2: 'Space',
    3: 'Power',
    4: 'Encryption',
    5: 'Hockey',
    6: 'Politics',
    7: 'Government',
    8: 'Money',
    9: 'Religion',
    10: 'Computers',
    11: 'Guns',
    12: 'General',
    13: 'Sales',
    14: 'Disk Drive',
    15: 'Files',
    16: 'Conversation',
    17: 'Medical',
    18: 'Technology',
    19: 'Mideast',
    20: 'Sports',
    21: 'Christianity',
    22: 'Automotive',
    23: 'Gaming',
    24: 'Queries',
    25: 'Cars',
    26: 'Law',
    27: 'Armenian Conflict',
    28: 'Internet',
    29: 'Windows'
}
topic_names = dct # Assign the new dictionary to topic_names

count_vectorizer = joblib.load('/content/Models/count_vectorizer.joblib')
label_encoder = joblib.load('/content/Models/label_encoder.joblib')

In [10]:
def clean_text(text):
    # Remove line breaks and lowercase
    text = re.sub(r'\n', ' ', text)
    text = text.lower()

    # Expand common contractions
    contractions = {
        "don't": "do not",
        "didn't": "did not",
        "doesn't": "does not",
        "can't": "can not",
        "won't": "will not",
        "it's": "it is",
        "i'm": "i am",
        "they're": "they are",
        "we're": "we are",
        "isn't": "is not",
        "aren't": "are not",
        "wasn't": "was not",
        "weren't": "were not",
        "hasn't": "has not",
        "haven't": "have not",
        "hadn't": "had not"
    }

    for c, expanded in contractions.items():
        text = text.replace(c, expanded)

    # Handling numerical values
    text = re.sub(r'\b(19|20)\d{2}\b', ' year ', text)
    text = re.sub(r'\b\d+(\.\d+)?%\b', ' percent ', text)
    text = re.sub(r'\$\d+(\.\d+)?', ' money ', text)
    text = re.sub(r'\b\d+\b', '', text)

    # Remove punctuation
    text = re.sub(r'[^\w\s]', ' ', text)

    # Tokenize
    txt = text.split()

    # Remove stopwords
    txt = [w for w in txt if w not in STOP_WORDS]

    # Remove tokens containing digits
    txt = [w for w in txt if not any(char.isdigit() for char in w)]

    # Remove short tokens
    txt = [w for w in txt if len(w) > 2]

    # Lemmatization
    txt = [lemmatizer.lemmatize(w) for w in txt]

    text = ' '.join(txt)

    # Remove extra spaces
    text = re.sub(r'\s+', ' ', text).strip()

    return text

def clean_for_summary(text):
    text = re.sub(r'\n+', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()

    return text

#Summary

In [12]:
def extract_lsa_summary(text, sentences_count=3, language="english"):
    """
    Generates an extractive summary of the given text using the LSA algorithm.

    Args:
        text (str): The input text to summarize.
        sentences_count (int): The desired number of sentences in the summary.
        language (str): The language of the text (e.g., "english").

    Returns:
        str: The summarized text.
    """
    if not isinstance(text, str) or not text.strip():
        return ""

    parser = PlaintextParser.from_string(text, Tokenizer(language))

    actual_num_sentences = len(parser.document.sentences)
    total_words_in_document = len(parser.document.words)

    # Initial effective sentences count based on requested and actual sentences
    effective_sentences_count = min(sentences_count, actual_num_sentences)

    # Further adjust effective_sentences_count based on total words in document.
    # The LSA algorithm needs at least as many words as sentences to function properly.
    # This directly addresses the "words_count < sentences_count" warning in sumy.
    if effective_sentences_count > total_words_in_document:
        effective_sentences_count = total_words_in_document

    # If, after all adjustments, effective_sentences_count becomes 0 or less,
    # it means the text is too short or problematic for LSA, so return an empty string.
    if effective_sentences_count <= 0:
        return ""

    stemmer = Stemmer(language)
    summarizer = LsaSummarizer(stemmer)
    summarizer.stop_words = get_stop_words(language)

    summary_sentences = summarizer(parser.document, sentences_count=effective_sentences_count)
    return " ".join([str(sentence) for sentence in summary_sentences])

In [13]:
def generate_summaries_t5(texts, model_name = "facebook/bart-large-cnn", batch_size=16, max_summary_length=512, min_summary_length=15, num_beams=4):
    print(f"DEBUG: generate_summaries_t5 called with max_summary_length={max_summary_length}, num_beams={num_beams}")

    # Re-initialize tokenizer and model here to ensure they are defined
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    # model_name="google/flan-t5-base"
    model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

    # Ensure GPU is used if available
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    model.to(device)

    results = []

    # Process texts in batches
    for i in range(0, len(texts), batch_size):
        # Corrected: use batch_size for slicing in the else branch
        batch_texts = texts[i:i+batch_size].tolist() if hasattr(texts, 'tolist') else texts[i:i+batch_size]
        print(f"DEBUG: Input text for summarization batch ({i}-{i+len(batch_texts)-1}): {batch_texts}")

        # Prefix input for summarization
        input_texts = ["Paraphrase and summarize the following text in new sentences: " + str(text) for text in batch_texts]

        # Tokenize and generate
        inputs = tokenizer(input_texts, return_tensors='pt', max_length=512,
                           truncation=True, padding=True).to(device)
        with torch.no_grad():
            summary_ids = model.generate(
                inputs['input_ids'],
                max_length=max_summary_length,
                min_length=min_summary_length,
                num_beams=num_beams,
                early_stopping=True,
                repetition_penalty=2.0 # Added repetition penalty
            )

        # Decode and store
        summaries = tokenizer.batch_decode(summary_ids, skip_special_tokens=True)
        results.extend(summaries)
        print(f"DEBUG: Generated summaries batch ({i}-{i+len(batch_texts)-1}): {summaries}")

    return results

#App

In [14]:
app = Flask(__name__)

ngrok.kill()

ngrok.set_auth_token(YOUR_NGROK_AUTH_TOKEN)

PORT = 5000
public_url = ngrok.connect(PORT)
print(f' * ngrok tunnel available at: {public_url}')
print(f'You can access your HTML page at: {public_url}')

@app.route('/')
def index():
    return send_file('index.html')

@app.route('/styles.css')
def serve_css():
    return send_file('styles.css')

@app.route('/sentiment_per_topic_image')
def serve_sentiment_image():
    image_path = '/content/sentiment per topic.png'
    if os.path.exists(image_path):
        return send_file(image_path, mimetype='image/png')
    else:
        return jsonify({"error": "Sentiment per topic image not found."}), 404

@app.route('/analyze', methods=['POST'])
def analyze():
    input_text = None
    is_file_input = False # Flag to track if input is from file

    if 'file' in request.files:
        file = request.files['file']
        if file.filename != '':
            is_file_input = True # Set flag to True if input is from file
            file_extension = file.filename.split('.')[-1].lower()
            if file_extension == 'txt':
                input_text = file.read().decode('utf-8')
            elif file_extension == 'pdf':
                try:
                    reader = PyPDF2.PdfReader(io.BytesIO(file.read()))
                    input_text = ""
                    for page_num in range(len(reader.pages)):
                        input_text += reader.pages[page_num].extract_text() + "\n"
                except Exception as e:
                    return jsonify({"error": f"Failed to process PDF file: {e}"}), 400
            else:
                return jsonify({"error": "Unsupported file type. Only .txt and .pdf are allowed."}), 400
    elif 'text' in request.form:
        input_text = request.form.get('text')

    if not input_text or input_text.strip() == "":
        return jsonify({"error": "No text or valid file provided for analysis."}), 400

    raw_text_for_sentiment = input_text

    cleaned_text_for_topic = clean_text(input_text)
    text_for_summarization = clean_for_summary(input_text)

    X = count_vectorizer.transform([cleaned_text_for_topic])
    topic = lda_model.transform(X).argmax()
    topic_name = topic_names[topic]

    sentences = sent_tokenize(raw_text_for_sentiment)
    all_sentiments = []
    for sentence in sentences:
        if sentence.strip():
            sentiment_scores = sentiment_model.polarity_scores(sentence)
            compound_score = sentiment_scores['compound']
            if compound_score >= 0.05:
                all_sentiments.append('positive')
            elif compound_score <= -0.05:
                all_sentiments.append('negative')
            else:
                all_sentiments.append('neutral')

    if not all_sentiments:
        all_sentiments = ['neutral']

    abstractive_summary = generate_summaries_t5([text_for_summarization])[0]

    extractive_sentences_count = 2 if is_file_input else 1
    extractive_summary = extract_lsa_summary(text_for_summarization, sentences_count=extractive_sentences_count)

    words = [word.lower() for word in word_tokenize(cleaned_text_for_topic) if word.isalpha() and word not in STOP_WORDS]
    word_freq = Counter(words)
    top_words = word_freq.most_common(10)

    bigrams = list(nltk.bigrams(words))
    bigram_freq = Counter(bigrams)
    top_bigrams = [{"bigram": " ".join(bg), "count": count} for bg, count in bigram_freq.most_common(10)]

    sentence_lengths = [len(word_tokenize(s)) for s in sentences if s.strip()]
    if sentence_lengths:
        max_len = max(sentence_lengths)
        bins = list(range(0, max_len + 5, 5))
        if not bins or bins[-1] < max_len:
            bins.append(max_len + 1)
        sentence_len_dist = Counter()
        for length in sentence_lengths:
            bin_found = False;
            for i in range(len(bins) - 1):
                if bins[i] <= length <= bins[i+1]:
                    bin_label = f"{bins[i]}-{bins[i+1]}"
                    sentence_len_dist[bin_label] += 1
                    bin_found = True
                    break
            if not bin_found:
                if len(bins) > 0 and length > bins[-1]:
                    bin_label = f">{bins[-1]}"
                    sentence_len_dist[bin_label] += 1
                elif length > 0:
                    sentence_len_dist["Other"] += 1

        sorted_labels = sorted(sentence_len_dist.keys(), key=lambda x: int(x.split('-')[0].replace('>', '')) if x.startswith('>') else int(x.split('-')[0]))
        sentence_length_data = [{'label': label, 'count': sentence_len_dist[label]} for label in sorted_labels]
    else:
        sentence_length_data = []

    all_words = [len(word) for word in word_tokenize(raw_text_for_sentiment) if word.isalpha()]
    if all_words:
        max_word_len = max(all_words)
        bins_word = list(range(1, max_word_len + 3, 2))
        if not bins_word or bins_word[-1] < max_word_len:
            bins_word.append(max_word_len + 1)

        word_len_dist = Counter()
        for length in all_words:
            bin_found = False
            for i in range(len(bins_word) - 1):
                if bins_word[i] <= length <= bins_word[i+1]:
                    bin_label = f"{bins_word[i]}-{bins_word[i+1]}"
                    word_len_dist[bin_label] += 1
                    bin_found = True
                    break
            if not bin_found:
                if len(bins_word) > 0 and length > bins_word[-1]:
                    bin_label = f">{bins[-1]}"
                    word_len_dist[bin_label] += 1
                elif length > 0:
                    word_len_dist["Other"] += 1

        sorted_labels_word = sorted(word_len_dist.keys(), key=lambda x: int(x.split('-')[0].replace('>', '')) if x.startswith('>') else int(x.split('-')[0]))
        word_length_data = [{'label': label, 'count': word_len_dist[label]} for label in sorted_labels_word]
    else:
        word_length_data = []

    flesch_reading_ease = textstat.flesch_reading_ease(raw_text_for_sentiment)
    flesch_kincaid_grade = textstat.flesch_kincaid_grade(raw_text_for_sentiment)

    return jsonify({
        "topics": [topic_name],
        "sentiments": all_sentiments,
        "abstractive_summary": abstractive_summary,
        "extractive_summary": extractive_summary,
        "top_words": top_words,
        "top_bigrams": top_bigrams,
        "sentence_length_distribution": sentence_length_data,
        "word_length_distribution": word_length_data,
        "readability": {
            "flesch_reading_ease": flesch_reading_ease,
            "flesch_kincaid_grade": flesch_kincaid_grade
        }
    })

@app.route('/get_20newsgroup_data', methods=['GET'])
def get_20newsgroup_data():
    if df_topics is None or df_sentiment_dataset is None:
        return jsonify({"error": "20 Newsgroup datasets not loaded."}), 500

    df_merged = pd.merge(
        df_sentiment_dataset,
        df_topics[['text', 'lda_simplified_topic_name']],
        on='text',
        how='left'
    )

    topic_counts = df_topics['lda_simplified_topic_name'].value_counts().to_dict()

    sentiment_counts = df_merged['sentiment_category'].value_counts().to_dict()

    return jsonify({
        "topic_distribution": topic_counts,
        "overall_sentiment_distribution": sentiment_counts
    })

app.run(port=PORT)

 * ngrok tunnel available at: NgrokTunnel: "https://f3a2-34-74-80-215.ngrok-free.app" -> "http://localhost:5000"
You can access your HTML page at: NgrokTunnel: "https://f3a2-34-74-80-215.ngrok-free.app" -> "http://localhost:5000"
 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5000
INFO:werkzeug:Press CTRL+C to quit
INFO:werkzeug:127.0.0.1 - - [11/Apr/2026 13:54:09] "GET / HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [11/Apr/2026 13:54:10] "GET /styles.css HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [11/Apr/2026 13:54:10] "GET /sentiment_per_topic_image HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [11/Apr/2026 13:54:10] "GET /favicon.ico HTTP/1.1" 404 -


DEBUG: generate_summaries_t5 called with max_summary_length=512, num_beams=4


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
Please make sure the generation config includes `forced_bos_token_id=0`. 


Loading weights:   0%|          | 0/511 [00:00<?, ?it/s]

...

DEBUG: Generated summaries batch (0-0): ['Paraphrase and summarize the following text in new sentences. Nature is a vital, nurturing force—our "mother" It acts as a bridge between the physical surroundings and our inner life.']
DEBUG: generate_summaries_t5 called with max_summary_length=512, num_beams=4


Loading weights:   0%|          | 0/511 [00:00<?, ?it/s]

DEBUG: Input text for summarization batch (0-0): ['Nature is a vital, nurturing force—our "mother"—offering sustenance through air, water, and food while providing a serene escape from daily life. It acts as a bridge between the physical surroundings and our inner life, acting as a precious gift that requires protection against pollution and urbanization. Nature is the most precious gift to humanity, functioning as a silent, nurturing mother that provides everything necessary for survival, including fresh air, water, and food. It encompasses the entire environment, from towering mountains and vast oceans to the smallest creatures and plants. Being in nature fosters a deep sense of peace and calm, offering a mental escape from the busy modern world. However, this vital balance is being threatened by human activities like pollution and industrialization. It is our responsibility to cherish, protect, and conserve nature to ensure a sustainable and beautiful future for generations to come.

INFO:werkzeug:127.0.0.1 - - [11/Apr/2026 14:13:00] "POST /analyze HTTP/1.1" 200 -


DEBUG: Generated summaries batch (0-0): ['Paraphrase and summarize the following text in new sentences. Nature is a vital, nurturing force—our "mother"—offering sustenance through air, water, and food while providing a serene escape from daily life. It is our responsibility to cherish, protect, and conserve nature to ensure a sustainable and beautiful future for generations to come.']
